In [2]:
import yaml
import os
import pandas as pd
import numpy as np
import time
import os
import tushare as ts
import tqdm

def load_config():
    with open("config.yaml", "r") as f:
        config = yaml.safe_load(f)
    return config
config = load_config()
TUSHARE_API_KEY = config["api"]["tushare_api_key"]
pd.set_option('display.max_columns', None)
ts.set_token(TUSHARE_API_KEY)
pro = ts.pro_api(TUSHARE_API_KEY)

In [3]:
def extract_code_and_weight_from_official_file(file_path):
    df = pd.read_excel(file_path,dtype={'成份券代码Constituent Code': str,'指数代码 Index Code':str})
    index_name = str(df["指数名称 Index Name"][0])
    weight_start_date = str(df["日期Date"][0])
    index_code = str(df["指数代码 Index Code"][0])
    if index_code[0] == '0':
        index_code = index_code + '.SH'
    if index_code[0] == '3':
        index_code = index_code + '.SZ'
    if index_code[0] == '8':
        index_code = index_code + '.BJ'
    print(f'该文件为{index_name}的官方权重文件,指数代码为{index_code},权重发布日期为{weight_start_date}')
    # 创建新的df，将原始官网文件中的成份券代码和权重提取出来
    new_df = df[['成份券代码Constituent Code','权重(%)weight']]
    new_df.columns = ['code',f'{weight_start_date}weight']
    # 给纯数字代码加上后缀
    for index in range(new_df.shape[0]):
        code = new_df.iloc[index]['code']
        if code[0] == '0' or code[0] == '3':
            new_code = code + '.SZ'
        elif code[0] == '6' or code[0] == '9':
            new_code = code + '.SH'
        else:
            print(f'{index}的后缀未知')
            raise Exception
        new_df.at[index, 'code'] = new_code
    # new_df.to_csv(os.path.join( f'./ref/project_/cal_index_data/{index_code}_{weight_start_date} - 副本.csv'), index=False)
    return index_name, index_code,weight_start_date,new_df
# file_path中的文件为中证官网下载文件
# file_path = r"C:/Users/18268/Downloads/000852closeweight (1).xls"
# file_path = r"C:/Users/18268/Downloads/000905closeweight.xls"
# file_path = r"C:/Users/18268/Downloads/000903closeweight.xls"
# file_path = r"C:/Users/18268/Downloads/000901closeweight.xls"
# file_path = r"C:/Users/18268/Downloads/000300closeweight.xls"
file_path = r"D:/Downloads/000852closeweight (3).xls"
index_name,index_code,weight_start_date,new_df = extract_code_and_weight_from_official_file(file_path)
new_df

该文件为中证1000的官方权重文件,指数代码为000852.SH,权重发布日期为20260227


,code,20260227weight
0,000012.SZ,0.073
1,000019.SZ,0.025
2,000028.SZ,0.053
3,000029.SZ,0.066
4,000030.SZ,0.030
...,...,...
995,688776.SH,0.065
996,688779.SH,0.142
997,688789.SH,0.099
998,688798.SH,0.079


In [4]:
# 批量添加收盘价
# 有些日期不交易，获取前一个交易日
def get_prev_trade_date(date):
    prev_day =(pd.to_datetime(date) - pd.Timedelta(days=1)).strftime('%Y%m%d')
    while True:
        if int(pro.trade_cal(exchange='SSE', start_date=prev_day, end_date=prev_day)['is_open'][0]) == 1:
            return prev_day
        else :
            prev_day =(pd.to_datetime(prev_day) - pd.Timedelta(days=1)).strftime('%Y%m%d')

# 给新的df添加收盘价并保存csv文件
def add_closePrice_and_write_to_csv(index_name,index_code,weight_start_date,new_df):
    # code_df = pd.read_csv(os.path.join('./ref/project_/cal_index_data', f'{index_code}_{weight_start_date} - 副本.csv'))
    code_df = new_df
    code_list = list(code_df["code"])
    code_list_ = ",".join(code_list)
    single_daily_info_df = pro.daily(ts_code=code_list_, trade_date=weight_start_date)
    single_daily_info_df
    closePrice_dict = dict(zip(single_daily_info_df['ts_code'], single_daily_info_df['close']))
    missing_codes = set(code_list) - set(list(single_daily_info_df['ts_code']))
    for code in missing_codes:
        prev_trade_date = get_prev_trade_date(weight_start_date)
        df = pro.daily(ts_code=code, start_date=prev_trade_date, end_date=prev_trade_date)
        while len(df) == 0:
            prev_trade_date = get_prev_trade_date(prev_trade_date)
            df = pro.daily(ts_code=code, start_date=prev_trade_date, end_date=prev_trade_date)
        closePrice_dict[code] = float(df['close'][0])
    closePrice_dict
    code_df[f'{weight_start_date}close'] = code_df['code'].map(closePrice_dict)

    if not os.path.exists(f'./ref/project_/cal_index_data/{index_name}_{index_code}'):
        os.makedirs(f'./ref/project_/cal_index_data/{index_name}_{index_code}')
    
    code_df.to_csv(os.path.join(f'./ref/project_/cal_index_data/{index_name}_{index_code}', f'{index_code}_{weight_start_date} - 副本.csv'),index=False)
    return code_df
code_df = add_closePrice_and_write_to_csv(index_name,index_code,weight_start_date,new_df)
code_df

Exception: 您的token不对，请确认。